# 04 · Essentia scores (run in Google Colab)

`essentia-tensorflow` only installs on Linux, so this notebook runs in **Google Colab**; everything else in
the project runs locally.

For every track it computes Discogs-EffNet embeddings and four classifiers on top of them:

| prefix in the CSV | model | classes |
|---|---|---|
| `genre__` | `genre_discogs400-discogs-effnet-1` | 400 Discogs styles |
| `instr__` | `mtg_jamendo_instrument-discogs-effnet-1` | 40 instruments |
| `mood__` | `mtg_jamendo_moodtheme-discogs-effnet-1` | 56 moods / themes |
| `voice__` | `voice_instrumental-discogs-effnet-1` | instrumental, voice |

Each activation is aggregated over time as `__mean` and `__max`, e.g. `instr__piano__mean`.

**Before running:** upload the 256 mp3 files (`data/raw/audio/` of the project) to Google Drive and set
`AUDIO_DIR` below. **Result:** `essentia_scores.csv` in Drive → copy it to `data/processed/` of the project
and run `python scripts/run_validation.py` locally. The loop is resumable: if Colab disconnects, run the
notebook again and it continues where it stopped.

In [ ]:
# ---- set these paths (Google Drive is mounted at /content/drive) ----
DRIVE_DIR = "/content/drive/MyDrive/ai-music-validator"   # a folder in your Google Drive
AUDIO_DIR = f"{DRIVE_DIR}/audio"                            # the 256 .mp3 files (subfolders are fine)
OUT_CSV = f"{DRIVE_DIR}/essentia_scores.csv"                # the result (written every SAVE_EVERY tracks)
MODEL_DIR = "/content/essentia_models"                      # models are downloaded here, not to Drive
SAVE_EVERY = 10

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Install essentia-tensorflow

In [ ]:
import subprocess
import sys

print("Python", sys.version.split()[0])
result = subprocess.run([sys.executable, "-m", "pip", "install", "essentia-tensorflow"], capture_output=True, text=True)
print(result.stdout[-1500:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError(
        f"pip could not install essentia-tensorflow for Python {sys.version_info.major}.{sys.version_info.minor} "
        "(see the pip error above). 'No matching distribution found' means there is no wheel for this Python "
        "version - check the supported versions at https://pypi.org/project/essentia-tensorflow/#files")

import essentia

print("essentia", essentia.__version__)

## 3. Download the models

From [essentia.upf.edu/models](https://essentia.upf.edu/models/): the `.pb` graph and its `.json` metadata.
Input / output node names and class labels are read from the `.json` files.

In [ ]:
import json
import urllib.request
from pathlib import Path

BASE = "https://essentia.upf.edu/models"
MODELS = {  # model name: (folder on the server, column prefix in the CSV; None = the embedding model)
    "discogs-effnet-bs64-1": ("feature-extractors/discogs-effnet", None),
    "genre_discogs400-discogs-effnet-1": ("classification-heads/genre_discogs400", "genre"),
    "mtg_jamendo_instrument-discogs-effnet-1": ("classification-heads/mtg_jamendo_instrument", "instr"),
    "mtg_jamendo_moodtheme-discogs-effnet-1": ("classification-heads/mtg_jamendo_moodtheme", "mood"),
    "voice_instrumental-discogs-effnet-1": ("classification-heads/voice_instrumental", "voice"),
}

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
meta = {}
for name, (folder, prefix) in MODELS.items():
    for ext in (".pb", ".json"):
        path = Path(MODEL_DIR) / f"{name}{ext}"
        if not path.exists():
            urllib.request.urlretrieve(f"{BASE}/{folder}/{name}{ext}", path)
    meta[name] = json.load(open(Path(MODEL_DIR) / f"{name}.json"))


def node_names(m, purpose):
    """(input node, output node with this output_purpose) from a model's .json metadata."""
    input_name = m["schema"]["inputs"][0]["name"]
    output_name = next(o["name"] for o in m["schema"]["outputs"] if o.get("output_purpose") == purpose)
    return input_name, output_name


for name, (folder, prefix) in MODELS.items():
    purpose = "embeddings" if prefix is None else "predictions"
    print(f"{name}: input / output = {node_names(meta[name], purpose)}, {len(meta[name].get('classes', []))} classes")

## 4. Load the models

In [ ]:
from essentia.standard import MonoLoader, TensorflowPredict2D, TensorflowPredictEffnetDiscogs

EMBEDDING_MODEL = "discogs-effnet-bs64-1"
input_name, output_name = node_names(meta[EMBEDDING_MODEL], "embeddings")
embedder = TensorflowPredictEffnetDiscogs(graphFilename=f"{MODEL_DIR}/{EMBEDDING_MODEL}.pb",
                                          input=input_name, output=output_name)
heads = {}  # prefix -> (classifier, class labels)
for name, (folder, prefix) in MODELS.items():
    if prefix is None:
        continue
    input_name, output_name = node_names(meta[name], "predictions")
    heads[prefix] = (TensorflowPredict2D(graphFilename=f"{MODEL_DIR}/{name}.pb", input=input_name, output=output_name),
                     meta[name]["classes"])
print({prefix: len(classes) for prefix, (_, classes) in heads.items()})

## 5. Scores for every track (resumable)

In [ ]:
import time

import numpy as np
import pandas as pd


def track_scores(path):
    """Mean and max over time of every class activation for one audio file."""
    audio = MonoLoader(filename=str(path), sampleRate=16000, resampleQuality=4)()
    embeddings = embedder(audio)  # (n_patches, 1280)
    row = {}
    for prefix, (model, classes) in heads.items():
        act = np.asarray(model(embeddings))  # (n_patches, n_classes)
        for i, label in enumerate(classes):
            row[f"{prefix}__{label}__mean"] = float(act[:, i].mean())
            row[f"{prefix}__{label}__max"] = float(act[:, i].max())
    return row


files = sorted(Path(AUDIO_DIR).rglob("*.mp3"))
print(f"{len(files)} mp3 files under {AUDIO_DIR}")
assert files, "no mp3 files found - check AUDIO_DIR"

done = pd.read_csv(OUT_CSV) if Path(OUT_CSV).exists() else pd.DataFrame()
done_ids = set(done["track_id"]) if len(done) else set()
todo = [f for f in files if f.stem not in done_ids]
print(f"{len(done_ids)} tracks already in {OUT_CSV}, {len(todo)} to go")

rows, failed, t0 = [], [], time.time()
for i, path in enumerate(todo, 1):
    try:
        rows.append({"track_id": path.stem, **track_scores(path)})
    except Exception as e:  # keep going, list the failures at the end
        failed.append((path.name, repr(e)))
    if i % SAVE_EVERY == 0 or i == len(todo):
        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True) if len(done) else pd.DataFrame(rows)
        done.to_csv(OUT_CSV, index=False)
        rows = []
        print(f"[{i}/{len(todo)}] saved {len(done)} tracks, {(time.time() - t0) / i:.1f} s per track", flush=True)

print(f"done: {len(done)} tracks in {OUT_CSV}")
if failed:
    print(f"{len(failed)} failed:", failed)

## 6. Quick check

In [ ]:
scores = pd.read_csv(OUT_CSV).set_index("track_id")
print(f"{scores.shape[0]} tracks, {scores.shape[1]} columns ({scores.shape[1] // 2} labels x mean / max)")
genre_cols = [c for c in scores.columns if c.startswith("genre__") and c.endswith("__mean")]
for tid in scores.index[:5]:
    top = scores.loc[tid, genre_cols].sort_values(ascending=False).head(3)
    print(f"{tid}: voice {scores.loc[tid, 'voice__voice__mean']:.2f} | top genres:",
          ", ".join(f"{c[len('genre__'):-len('__mean')]} ({v:.2f})" for c, v in top.items()))

## Next

1. Download `essentia_scores.csv` from your Drive folder (`DRIVE_DIR`).
2. Put it at `data/processed/essentia_scores.csv` in the project.
3. Run `python scripts/run_validation.py` locally: section 10 of `results/summary.md` then compares CLAP,
   Essentia and their average (without the file, that section is skipped).